# Reconhecimento de Gestos com Webcam, OpenCV e MediaPipe
Este notebook demonstra como realizar o reconhecimento de gestos em tempo real utilizando a sua webcam, o OpenCV (`cv2`) e a biblioteca de visão computacional MediaPipe do Google.

In [5]:
# Instalando as dependências necessárias
!pip install opencv-python mediapipe scikit-learn



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import urllib.request
import os

# O MediaPipe requer um modelo treinado específico para reconhecimento de gestos.
model_path = 'gesture_recognizer.task'
if not os.path.exists(model_path):
    print("Baixando o modelo de gestos...")
    url = 'https://storage.googleapis.com/mediapipe-models/gesture_recognizer/gesture_recognizer/float16/1/gesture_recognizer.task'
    urllib.request.urlretrieve(url, model_path)
    print("Modelo baixado com sucesso!")
else:
    print("O modelo já existe localmente.")

O modelo já existe localmente.


In [7]:
import cv2
import mediapipe as mp

# Inicializando os módulos correspondentes do MediaPipe
BaseOptions = mp.tasks.BaseOptions
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Configurando as opções para o reconhecedor
options = GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.IMAGE, # Modo de imagem
    num_hands=2 # Permite que até 2 mãos simultâneas sejam identificadas!
)

recognizer = GestureRecognizer.create_from_options(options)
print("Reconhecedor de gestos inicializado!")

Reconhecedor de gestos inicializado!


In [8]:
# Captura de Vídeo e Reconhecimento em Tempo Real
import time
import matplotlib.pyplot as plt
import pickle
import numpy as np

# Carregar o modelo treinado de gestos customizados
with open('modelo_gestos.pkl', 'rb') as f:
    clf = pickle.load(f)

# Ferramentas utilitárias de desenho fornecidas pelas tasks atualizadas do MediaPipe
mp_hands = mp.tasks.vision.HandLandmarksConnections
mp_drawing = mp.tasks.vision.drawing_utils
mp_drawing_styles = mp.tasks.vision.drawing_styles

# Inicializar a Webcam (0 é a câmera padrão)
cap = cv2.VideoCapture(0)

# Constantes para o texto na tela
MARGIN = 10 
ROW_SIZE = 10 
FONT_SIZE = 1
FONT_THICKNESS = 1
TEXT_COLOR = (0, 255, 0) # Verde

print("Abrindo a Webcam... Pressione a tecla 'q' na janela de vídeo para sair.")

try:
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("Captura da webcam falhou ou arquivo terminou. Saindo...")
            break

        # O OpenCV lê a imagem como BGR, mas o MediaPipe espera RGB
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_image)

        # Realiza o reconhecimento de gestos neste frame
        recognition_result = recognizer.recognize(mp_image)

        if recognition_result.hand_landmarks:
            # Percorre as mãos detectadas
            for i, hand_landmarks in enumerate(recognition_result.hand_landmarks):
                # 1. Predição do MediaPipe (Gestos Originais)
                mp_category = "None"
                mp_score = 0.0
                if recognition_result.gestures and i < len(recognition_result.gestures):
                    if recognition_result.gestures[i]:
                        mp_category = recognition_result.gestures[i][0].category_name
                        mp_score = round(recognition_result.gestures[i][0].score, 2)

                # 2. Predição do nosso Modelo Customizado
                row = []
                for landmark in hand_landmarks:
                    row.extend([landmark.x, landmark.y, landmark.z])
                custom_category = clf.predict([row])[0]
                custom_prob = round(np.max(clf.predict_proba([row])[0]), 2)
                
                # 3. Lógica para combinar os dois!
                # O MediaPipe geralmente retorna 'None' ou tem uma pontuação muito baixa para gestos que não conhece
                if mp_category != "None" and mp_category != "" and mp_score > 0.5:
                    result_text = f'Original: {mp_category} ({mp_score})'
                    TEXT_COLOR = (255, 255, 0) # Ciano para original
                else:
                    result_text = f'Novo: {custom_category} ({custom_prob})'
                    TEXT_COLOR = (0, 255, 0) # Verde para os seus novos
                
                # Desenha o texto do gesto na tela
                text_location = (MARGIN, MARGIN + ROW_SIZE + 30 + (i * 40))
                cv2.putText(image, result_text, text_location, cv2.FONT_HERSHEY_DUPLEX,
                            FONT_SIZE, TEXT_COLOR, FONT_THICKNESS)
                
                # Desenhando os landmarks
                mp_drawing.draw_landmarks(
                    image,
                    hand_landmarks,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),
                    mp_drawing_styles.get_default_hand_connections_style()
                )

        # Mostrando o frame com os resultados 
        cv2.imshow('Reconhecimento de Gestos - MediaPipe', image)
        
        # Espera o clique da tecla 'q' para sair
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
except KeyboardInterrupt:
    print("Script interrompido pelo usuário.")
finally:
    # Liberar recursos quando finalizado ou abortado
    cap.release()
    cv2.destroyAllWindows()
    print("Webcam liberada. Prontinho!")


FileNotFoundError: [Errno 2] No such file or directory: 'modelo_gestos.pkl'